# Fase 02: Análisis Exploratorio de Datos (EDA)
**Objetivo:** Realizar análisis estadístico y visual de los datos en capa Bronze para entender distribuciones, correlaciones y patrones clínicos relevantes. Incluye análisis de readmisiones, top diagnósticos, top medicamentos y distribuciones de laboratorio.
**Entrada:** Archivos Parquet en `medicalproyect-raw/bronze/`.
**Salida:** Visualizaciones + logs de análisis en `medicalproyect-raw/logs/analisis/`.

## 1. Inicialización de Spark y Logger


In [ ]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

# ── Función helper: sube el log al Data Lake ──
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass

# ── Inicialización y Logger ──
# Configuración inicial: creamos Spark y el logger para poder procesar datos y registrar todo lo que hagamos.
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
from pyspark.ml.feature import Imputer
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import os
from datetime import datetime

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"


# Limpieza de handlers previos (evita duplicados al re-ejecutar)
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

# Configuración del Logger
log_filename = "pipeline_analisis.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.Analysis.EDA")

logger.info("=" * 60)
logger.info("INICIO DEL PIPELINE DE ANALISIS (CAPA BRONZE)")
logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)

spark = SparkSession.builder \
    .appName("Medical_Analysis_EDA") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

logger.info("SparkSession creada.")
logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 02 Exploracion Eda")


## 2. Carga desde Capa Bronze (Parquet)


In [ ]:
# ── Carga de Datos desde Capa Bronze (Parquet) ──
# Cargamos los datos desde la capa Bronze del Data Lake para empezar a analizarlos.
STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW  = "medicalproyect-raw"
CONTAINER_LOGS = "medicalproyect-logs"
base_path = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

logger.info("Cargando datos desde la capa Bronze (Parquet)...")

df_patients    = spark.read.parquet(f"{base_path}/bronze/patients")
df_outcomes    = spark.read.parquet(f"{base_path}/bronze/outcomes")
df_medications = spark.read.parquet(f"{base_path}/bronze/medications")
df_lab_results = spark.read.parquet(f"{base_path}/bronze/lab_results")
df_diagnoses   = spark.read.parquet(f"{base_path}/bronze/diagnoses")

logger.info(f"Patients:    {df_patients.count()} registros")
logger.info(f"Outcomes:    {df_outcomes.count()} registros")
logger.info(f"Medications: {df_medications.count()} registros")
logger.info(f"Lab Results: {df_lab_results.count()} registros")
logger.info(f"Diagnoses:   {df_diagnoses.count()} registros")

logger.info("")
logger.info("Vista previa de Patients:")
df_patients.show(5, truncate=False)

## 3. Estadísticas Básicas y Valores Nulos


In [ ]:
# ── Estadisticas Basicas y Valores Nulos ──
# Primera exploración: miramos cuántos datos tenemos y si hay valores nulos que limpiar.
logger.info("")
logger.info("── Dimension de las Tablas ──")
logger.info(f"Patients:    {df_patients.count()} filas, {len(df_patients.columns)} columnas")
logger.info(f"Outcomes:    {df_outcomes.count()} filas, {len(df_outcomes.columns)} columnas")
logger.info(f"Medications: {df_medications.count()} filas, {len(df_medications.columns)} columnas")
logger.info(f"Lab Results: {df_lab_results.count()} filas, {len(df_lab_results.columns)} columnas")
logger.info(f"Diagnoses:   {df_diagnoses.count()} filas, {len(df_diagnoses.columns)} columnas")

logger.info("")
logger.info("── Valores Distintos en Categorias Clave (Patients) ──")
cols_cat = ["sex", "smoking_status", "alcohol_use", "exercise_level", "insurance_type"]
for c in cols_cat:
    n_distinct = df_patients.select(F.countDistinct(F.col(c))).collect()[0][0]
    logger.info(f"  {c}: {n_distinct} valores distintos")

logger.info("")
logger.info("── Conteo de Valores Nulos (Patients) ──")
df_patients.select(
    [F.count(F.when(F.col(c).isNull(), F.col(c))).alias(c) for c in df_patients.columns]
).show(vertical=True)
# Subida parcial de log tras esta seccion
_subir_log("analisis/analisis_imputación", STORAGE_ACCOUNT)


## 4. Imputación de Nulos y Matriz de Correlación


In [ ]:
# ── Imputacion de Nulos (Mediana) ──
# Rellenamos los nulos con la mediana para no perder datos al calcular las correlaciones.
num_cols_patients = [
    f.name for f in df_patients.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != 'patient_id'
]

imputer = Imputer(
    inputCols=num_cols_patients,
    outputCols=num_cols_patients,
    strategy="median"
)
df_patients_clean = imputer.fit(df_patients).transform(df_patients)
logger.info("")
logger.info("Nulos imputados con la mediana estadistica.")

# ── Matriz de Correlacion de Variables Clinicas ──
logger.info("")
logger.info("Generando matriz de correlacion de variables clinicas...")

cols_corr = ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'charlson_index']
df_corr_pd = df_patients_clean.select(cols_corr).toPandas()

plt.figure(figsize=(8, 6))
sns.heatmap(df_corr_pd.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Interaccion Interna de Variables Clinicas (Correlacion)', fontsize=14)
plt.tight_layout()
plt.show()
# Subida parcial de log tras esta seccion
_subir_log("analisis/analisis_readmisiones", STORAGE_ACCOUNT)


## 5. Análisis de Readmisiones (Variable Objetivo)


In [ ]:
# ── Analisis de Readmisiones a 30 Dias ──
# Analizamos las readmisiones porque es nuestra variable objetivo, lo que queremos predecir.
logger.info("")
logger.info("── Readmisiones a 30 Dias ──")
df_outcomes.groupBy(F.col("readmitted_30d")) \
    .agg(
        F.count("*").alias("total_pacientes"),
        F.round(F.avg(F.col("length_of_stay_days")), 1).alias("estancia_media_dias"),
        F.round(F.avg(F.col("total_charges_usd")), 2).alias("coste_medio_usd"),
        F.round(F.avg(F.col("in_hospital_death")), 4).alias("tasa_mortalidad")
    ).show()
# Subida parcial de log tras esta seccion
_subir_log("analisis/analisis_top_diagnósticos", STORAGE_ACCOUNT)


## 6. Top Diagnósticos y Medicamentos


In [ ]:
# ── Top Diagnosticos y Medicamentos ──
# Revisamos los diagnósticos y medicamentos más frecuentes para entender el contexto clínico de los pacientes.
logger.info("")
logger.info("── Top 5 Diagnosticos Principales ──")
df_diagnoses.groupBy(F.col("primary_diagnosis")) \
    .count() \
    .orderBy(F.desc("count")) \
    .show(5, truncate=False)

logger.info("")
logger.info("── Top 5 Medicamentos con Menor Adherencia ──")
df_medications.groupBy(F.col("indication")) \
    .agg(F.round(F.avg(F.col("adherence_pct")), 2).alias("adherencia_media")) \
    .orderBy(F.asc("adherencia_media")) \
    .show(5, truncate=False)

logger.info("")
logger.info("── Top 5 Medicamentos mas Prescritos ──")
df_medications.groupBy(F.col("medication")) \
    .count() \
    .orderBy(F.desc("count")) \
    .show(5, truncate=False)
# Subida parcial de log tras esta seccion
_subir_log("analisis/analisis_distribución", STORAGE_ACCOUNT)


## 7. Distribución de Pruebas de Laboratorio Anormales


In [ ]:
# ── Distribucion de Pruebas de Laboratorio Anormales ──
# Vemos cómo se distribuyen los valores anormales de laboratorio para identificar patrones o outliers.
logger.info("")
logger.info("Generando boxplots de pruebas de laboratorio anormales...")

df_lab_abnormal = df_lab_results \
    .filter(F.col("is_abnormal") == 1) \
    .select(F.col("test_name"), F.col("value")) \
    .sample(fraction=0.1, seed=42) \
    .toPandas()

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_lab_abnormal, x="test_name", y="value")
plt.xticks(rotation=45)
plt.title('Distribucion y Atipicos en Pruebas de Laboratorio Anormales')
plt.tight_layout()
plt.show()

## 8. Distribución de Variables Categóricas


In [ ]:
# ── Distribucion de Variables Categoricas ──
# Analizamos la distribución de categorías para detectar si hay desbalanceo en los datos.
def analizar_categoria(df, nombre_columna):
    total_filas = df.count()
    df_resultado = df.groupBy(F.col(nombre_columna)) \
        .agg(F.count("*").alias("Cantidad")) \
        .withColumn("Porcentaje(%)", F.round((F.col("Cantidad") / total_filas) * 100, 2)) \
        .orderBy(F.desc("Cantidad"))

    logger.info("")
    logger.info(f"DISTRIBUCION DE: {nombre_columna.upper()}")
    df_resultado.show(truncate=False)

logger.info("")
logger.info("── Distribuciones Categoricas ──")
analizar_categoria(df_patients, "sex")
analizar_categoria(df_patients, "smoking_status")
analizar_categoria(df_patients, "insurance_type")
analizar_categoria(df_patients, "alcohol_use")
analizar_categoria(df_patients, "exercise_level")

logger.info("")
logger.info("── Distribucion de Diagnostico por Tipo de Visita ──")
df_diagnoses.groupBy(F.col("visit_type")) \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)
# Subida parcial de log tras esta seccion
_subir_log("analisis/analisis_persistencia", STORAGE_ACCOUNT)


## 9. Persistencia de Logs y Cierre


In [ ]:
# ── Persistencia de Logs y Cierre ──
# Guardamos el log en el Data Lake y cerramos Spark para liberar recursos.
logger.info("")
logger.info("Guardando archivo de logs en el Data Lake...")

try:
    from notebookutils import mssparkutils
    fecha_str = datetime.now().strftime('%Y%m%d_%H%M')
    log_destino = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/analisis/analisis_{fecha_str}.log"
    ruta_log_local = f"file://{os.path.abspath(log_filename)}"
    mssparkutils.fs.cp(ruta_log_local, log_destino)
    logger.info(f"Log persistido en: {log_destino}")
except Exception as e:
    logger.warning(f"No se pudo copiar el log al Data Lake: {e}")

logger.info("")
logger.info("=" * 60)
logger.info("PIPELINE DE ANALISIS FINALIZADO")
logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)

logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 02 Exploracion Eda")
spark.stop()
logger.info("SparkSession detenida.")